#### 1. Import libraries.

In [1]:
%cd ..

/Users/mateuszgrzyb/Projekty/algorithmic_trading


In [2]:
import joblib

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

from src.utils.tools import calculate_financial_ratios, top_k_score
from src.modelling.model_pipeline import FactorModelPipeline

TARGET = 'label_2000_100_252_pct_change'
MODEL_NAME = 'ols_20260611_1034'

#### 2. Load data.

In [3]:
tr = pd.read_feather('data/abt_clean/tr.feather')
va = pd.read_feather('data/abt_clean/va.feather')
te = pd.read_feather('data/abt_clean/te.feather')

In [4]:
features_to_model = ['price_to_sales', 
                     'roe']

#### 3. Load model.

In [5]:
pipeline = FactorModelPipeline.load(f'models/{MODEL_NAME}.joblib')

#### 4. Score data.

In [6]:
pred_tr = pipeline.predict(tr)
pred_va = pipeline.predict(va)
pred_te = pipeline.predict(te)

tr['pred'] = pred_tr
va['pred'] = pred_va
te['pred'] = pred_te

#### 7. Search for optimal "n".

##### 7.1. Validation dataset.

In [7]:
results = []
for n in range(1, 21):
    cohort_returns = []
    for quarter in va.date.unique():
        cohort_return = va[va.date == quarter]\
            .sort_values('pred', ascending=False)\
            .iloc[0:n][TARGET]\
            .mean()
        cohort_returns.append(cohort_return)

    # Agregacja metryk dla danego N
    avg_return = np.mean(cohort_returns)
    volatility = np.std(cohort_returns)
    worst_cohort = np.min(cohort_returns)
    sharpe_proxy = avg_return / volatility if volatility > 0 else 0
    
    results.append({
        'N': n,
        'Avg_Return': avg_return,
        'Volatility': volatility,
        'Worst_Drawdown': worst_cohort,
        'Sharpe': sharpe_proxy
    })
results_va = pd.DataFrame(results)

In [8]:
results_va.head(20)

,N,Avg_Return,Volatility,Worst_Drawdown,Sharpe
0,1,59.144921,168.886688,-59.117992,0.350205
1,2,62.400582,124.496630,-34.147428,0.501223
2,3,49.727284,83.764732,-21.446707,0.593654
3,4,41.333608,65.021546,-19.334807,0.635691
4,5,36.270229,54.016223,-20.157703,0.671469
5,6,34.164569,46.773159,-17.916360,0.730431
6,7,33.767343,45.872593,-17.009047,0.736111
7,8,30.773071,41.035825,-10.327290,0.749907
8,9,29.924492,38.958174,-10.679560,0.768118
9,10,30.646475,36.797858,-8.424022,0.832833


##### 7.2. Test dataset.

In [9]:
results = []
for n in range(1, 21):
    cohort_returns = []
    for quarter in te.date.unique():
        cohort_return = te[te.date == quarter]\
            .sort_values('pred', ascending=False)\
            .iloc[0:n][TARGET]\
            .mean()
        cohort_returns.append(cohort_return)

    # Agregacja metryk dla danego N
    avg_return = np.mean(cohort_returns)
    volatility = np.std(cohort_returns)
    worst_cohort = np.min(cohort_returns)
    sharpe_proxy = avg_return / volatility if volatility > 0 else 0
    
    results.append({
        'N': n,
        'Avg_Return': avg_return,
        'Volatility': volatility,
        'Worst_Drawdown': worst_cohort,
        'Sharpe': sharpe_proxy
    })
results_te = pd.DataFrame(results)

In [10]:
results_te.head(20)

,N,Avg_Return,Volatility,Worst_Drawdown,Sharpe
0,1,38.517092,28.356464,-13.121638,1.358318
1,2,81.863035,179.173396,-16.115251,0.456893
2,3,60.150804,118.061100,-0.096757,0.509489
3,4,45.542330,86.697639,-2.976337,0.525301
4,5,37.250571,71.317976,-6.238723,0.522317
5,6,31.575560,59.865935,-3.421254,0.527438
6,7,28.967601,53.850798,-3.249562,0.537923
7,8,26.741393,49.272427,-2.642213,0.542725
8,9,24.029356,44.158754,-2.809745,0.544158
9,10,23.039382,41.736045,-5.306597,0.552026


##### 7.3. Both.

In [11]:
va_te = pd.concat([va, te])
results = []
for n in range(1, 21):
    cohort_returns = []
    for quarter in va_te.date.unique():
        cohort_return = va_te[va_te.date == quarter]\
            .sort_values('pred', ascending=False)\
            .iloc[0:n]['label_2000_100_252_pct_change']\
            .mean()
        cohort_returns.append(cohort_return)

    # Agregacja metryk dla danego N
    avg_return = np.mean(cohort_returns)
    volatility = np.std(cohort_returns)
    worst_cohort = np.min(cohort_returns)
    sharpe_proxy = avg_return / volatility if volatility > 0 else 0
    
    results.append({
        'N': n,
        'Avg_Return': avg_return,
        'Volatility': volatility,
        'Worst_Drawdown': worst_cohort,
        'Sharpe': sharpe_proxy
    })
results_all = pd.DataFrame(results)

In [12]:
results_all.head(10)

,N,Avg_Return,Volatility,Worst_Drawdown,Sharpe
0,1,52.268978,139.203912,-59.117992,0.375485
1,2,68.888067,145.320854,-34.147428,0.474041
2,3,53.201791,96.684913,-21.446707,0.550259
3,4,42.736515,72.992904,-19.334807,0.585489
4,5,36.597010,60.339040,-20.157703,0.606523
5,6,33.301566,51.522991,-17.916360,0.646344
6,7,32.167429,48.730060,-17.009047,0.660115
7,8,29.429179,43.994266,-10.327290,0.668932
8,9,27.959447,40.860098,-10.679560,0.684273
9,10,28.110777,38.680920,-8.424022,0.726735


In [13]:
top_k_score(va, target=TARGET, k=5)

np.float64(36.27022916508307)

In [14]:
top_k_score(te, target=TARGET, k=5)

np.float64(37.250570925484475)